# GraphKV v5：五节点单卡调用测试

图：**4 → 1，4 → 3，5 → 2，5 → 3**。source = {4,5}，target = {1,2,3}。

请使用已有 GraphKV Python 内核，从上到下运行。模型路径已填好；通常只需修改下一格 `REPO_DIR`（必须指向含 `graph_kv_cache.py` 的目录）。本文件没有预填模型输出：运行后每步会显示真实结果，不能将示意或预期结果当成已执行结果。

核对的公开 v5 commit：`1af09232d361d8f529d951f35348664aad37d415`。

参考：[论文 v2 §3.1](https://arxiv.org/html/2506.07334v2#S3.SS1)、[v5 源码](https://github.com/zmsnhcy/GraphKV_refactor_v5/tree/1af09232d361d8f529d951f35348664aad37d415/GraphKV_refactor_v5)、[官方仓库](https://github.com/Graph-COM/GraphKV)。论文采用 source 共享位置、target 共享后一段位置，query 读取 source 和 target。这里调用 v5 本地模块，不走服务端。


## 1. 设置单卡与本地路径
必须在导入 torch / 初始化 CUDA 之前设置 GPU。如果当前内核已经使用过 CUDA，请重启内核再运行。`GPU_ID="0"` 表示物理 GPU 0；屏蔽后程序内对应 `cuda:0`。不使用自动多卡分配，不下载模型。


In [2]:
import os, sys
from pathlib import Path
GPU_ID = "3"
REPO_DIR = Path("/home/lyh/hcy_work/GraphKV_refactor_v5")  # 改为含两个 graph_kv 模块的目录
MODEL_PATH = Path("/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT")
BACKEND = "sdpa"  # 默认不依赖 flash-attn；也可用 eager / flash_attention_2
MAX_NEW_TOKENS = 80
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("CUDA 已初始化，请重启内核，再从本格运行。")
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# GitHub 仓库外层可能还有一层同名目录。
if not (REPO_DIR / "graph_kv_cache.py").is_file():
    nested = REPO_DIR / "GraphKV_refactor_v5"
    if (nested / "graph_kv_cache.py").is_file():
        REPO_DIR = nested
assert (REPO_DIR / "graph_kv_cache.py").is_file(), f"请修改 REPO_DIR: {REPO_DIR}"
assert (MODEL_PATH / "config.json").is_file(), f"找不到本地模型: {MODEL_PATH}"
sys.path.insert(0, str(REPO_DIR.resolve()))
print("模块目录:", REPO_DIR, "模型目录:", MODEL_PATH, "物理 GPU:", GPU_ID, sep="\n")


模块目录:
/home/lyh/hcy_work/GraphKV_refactor_v5
模型目录:
/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT
物理 GPU:
3


In [3]:
import json, inspect, gc, time
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from graph_kv_cache import GraphKVCache
from graph_kv_adapter import greedy_generate_graphkv, graphkv_next_logits, build_graphkv_from_ids
from IPython.display import display, HTML
assert transformers.__version__ == "4.50.0", "v5 adapter 要求 transformers==4.50.0，请选择原有兼容内核。"
assert torch.cuda.is_available(), "当前内核没有可用 CUDA"
assert torch.cuda.device_count() == 1, "单卡屏蔽未生效，请重启内核"
assert torch.cuda.is_bf16_supported(), "此测试默认使用 BF16"
print("torch:", torch.__version__, "transformers:", transformers.__version__)
print("实际导入:", inspect.getfile(GraphKVCache))
print("GPU:", torch.cuda.get_device_name(0))
def memory(label):
    free, total = torch.cuda.mem_get_info(0)
    print(f"{label}: free={free/2**30:.2f}/{total/2**30:.2f} GiB; "
          f"allocated={torch.cuda.memory_allocated()/2**30:.2f} GiB; "
          f"peak={torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
memory("加载前")


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1+cu121 transformers: 4.50.0
实际导入: /home/lyh/hcy_work/GraphKV_refactor_v5/graph_kv_cache.py
GPU: NVIDIA GeForce RTX 4090
加载前: free=23.26/23.64 GiB; allocated=0.00 GiB; peak=0.00 GiB


## 2. 手动构造节点文本与图
文本使用便于核对的虚构事实。4 提供 ALPHA 的材料，5 提供 BETA 的材料；1、2、3 分别使用 ALPHA、BETA、两者。图依赖由 `set_edges` 控制，文本本身不会创建注意力边。

块级可见关系：4/5 各自独立；1 可读 4；2 可读 5；3 可读 4 和 5。各块内部仍是因果注意力。prefix 仅在 query 阶段可见，不参与节点编码。


In [4]:
SOURCE_IDS = ["4", "5"]
TARGET_IDS = ["1", "2", "3"]
EDGES = [("4", "1"), ("4", "3"), ("5", "2"), ("5", "3")]
source_texts = {
    "4": "[Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.\n",
    "5": "[Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.\n",
}
target_texts = {
    "1": "[Document 1] Project ORION uses the material supplied by ALPHA.\n",
    "2": "[Document 2] Project LYRA uses the material supplied by BETA.\n",
    "3": "[Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.\n",
}
PREFIX = "<|user|>\nAnswer using the following documents. Give a short factual answer.\n"
QUERY = "\nQuestion: What material does ORION use, what material does LYRA use, and which two materials does NOVA combine?\n<|assistant|>\n"
assert set(source_texts).isdisjoint(target_texts)
assert all(s in source_texts and t in target_texts for s,t in EDGES)
print("边:", EDGES)
for n, text in {**source_texts, **target_texts}.items():
    print(n, "source" if n in SOURCE_IDS else "target", text)
print("target <- sources:", {t:[s for s,d in EDGES if d==t] for t in TARGET_IDS})
display(HTML("<pre>4 ─────► 1\n│\n└──────► 3 ◄──────┐\n                   │\n5 ─────► 2         │\n└──────────────────┘</pre>"))


边: [('4', '1'), ('4', '3'), ('5', '2'), ('5', '3')]
4 source [Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.

5 source [Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.

1 target [Document 1] Project ORION uses the material supplied by ALPHA.

2 target [Document 2] Project LYRA uses the material supplied by BETA.

3 target [Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.

target <- sources: {'1': ['4'], '2': ['5'], '3': ['4', '5']}


## 3. Tokenize 与位置规划（先覆盖全部节点长度）
`P = prefix token 数`，`L = 全部 source/target 最大 token 数`。
source 从 P 开始，target 从 P+L 开始，query 从 P+2L 开始。不能用物理 KV 总长度替代这些逻辑位置。所有块都不添加隐含 BOS、不 padding、不截断；保留仓库示例的 Tulu prompt。


In [5]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
config = AutoConfig.from_pretrained(str(MODEL_PATH), local_files_only=True)
assert config.model_type == "llama"
rope = config.rope_scaling or {}
assert rope.get("rope_type", rope.get("type", "default")) in ("default", "llama3", "linear")
def tokenize(text):
    return tokenizer(text, return_tensors="pt", add_special_tokens=False,
                     truncation=False).input_ids
node_inputs = {n:tokenize(t) for n,t in {**source_texts, **target_texts}.items()}
prefix_ids, query_ids = tokenize(PREFIX), tokenize(QUERY)
lengths = {n:int(x.shape[1]) for n,x in node_inputs.items()}
P = int(prefix_ids.shape[1])
L = max(lengths.values())
Q = int(query_ids.shape[1])
assert P + 2*L + Q + MAX_NEW_TOKENS <= config.max_position_embeddings
print("P, L, Q =", P, L, Q)
print("node | kind | tokens | logical range [start,end)")
for n in SOURCE_IDS + TARGET_IDS:
    start = P if n in SOURCE_IDS else P+L
    print(n, "source" if n in SOURCE_IDS else "target", lengths[n], (start,start+lengths[n]))
print("query logical:", (P+2*L, P+2*L+Q))
print("query physical start:", P+sum(lengths.values()))
print("Tulu markers:", {x:tokenizer.encode(x, add_special_tokens=False) for x in ["<|user|>","<|assistant|>"]})


P, L, Q = 17 21 31
node | kind | tokens | logical range [start,end)
4 source 20 (17, 37)
5 source 21 (17, 38)
1 target 15 (38, 53)
2 target 16 (38, 54)
3 target 18 (38, 56)
query logical: (59, 90)
query physical start: 107
Tulu markers: {'<|user|>': [27, 91, 882, 91, 29], '<|assistant|>': [27, 91, 78191, 91, 29]}


## 4. 加载本地 8B 模型到唯一可见的 GPU
BF16 的 8B 参数约占 15 GiB，另外需要缓存、激活及 CUDA 工作区；是否能装下取决于该卡当前空闲显存。示例文本较短，逐节点执行，batch=1。若 OOM，先释放同卡其他进程并重启内核；缩短生成不能解决模型权重本身不足的问题。


In [6]:
torch.manual_seed(2026)
torch.cuda.reset_peak_memory_stats()
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH), local_files_only=True, torch_dtype=torch.bfloat16,
    device_map={"": 0}, low_cpu_mem_usage=True, attn_implementation=BACKEND,
).eval()
devices = {str(p.device) for p in model.parameters()}
assert devices == {"cuda:0"}, devices
assert next(model.parameters()).dtype == torch.bfloat16
device = model.get_input_embeddings().weight.device
print("所有参数所在设备:", devices, "backend:", model.config._attn_implementation)
memory("模型加载后")


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.05s/it]

所有参数所在设备: {'cuda:0'} backend: sdpa
模型加载后: free=8.30/23.64 GiB; allocated=14.96 GiB; peak=14.96 GiB


## 5. 独立编码 prefix、source 4、source 5
这里显式调用底层类，便于观察每一步。传入的物理位置始终从 0 开始，source 的 RoPE 逻辑位置从 P 开始。`assume_positioned=True` 表示前向时已经设置正确位置，不再重旋转。


In [7]:
def seq_len(cache):
    return int(cache.key_cache[0].shape[-2])
def kv_bytes(cache):
    return sum(x.numel()*x.element_size() for xs in (cache.key_cache,cache.value_cache) for x in xs)
@torch.inference_mode()
def encode_independent(ids, logical_start):
    ids = ids.to(device)
    slots = torch.arange(ids.shape[1], device=device)
    out = model(ids, position_ids=(slots+logical_start).unsqueeze(0),
                cache_position=slots, attention_mask=torch.ones_like(ids),
                use_cache=True, return_dict=True)
    return out.past_key_values
gkv = GraphKVCache(source_position_start=P, chunk_span=L)
gkv.set_edges(EDGES)
cache = encode_independent(prefix_ids, 0)
gkv.set_prefix_cache(cache)
del cache
for n in SOURCE_IDS:
    cache = encode_independent(node_inputs[n], P)
    gkv.add_source(n, cache, assume_positioned=True)
    del cache
    m = gkv.meta[n]
    print(f"source {n}: length={m.token_count}, logical=[{m.logical_start},{m.logical_end}), "
          f"layer0 K={tuple(gkv.nodes[n].key_cache[0].shape)}, KV={kv_bytes(gkv.nodes[n])/2**20:.3f} MiB")
assert set(gkv.nodes) == {"4","5"}
memory("source 完成")


source 4: length=20, logical=[17,37), layer0 K=(1, 8, 20, 128), KV=2.500 MiB
source 5: length=21, logical=[17,38), layer0 K=(1, 8, 21, 128), KV=2.625 MiB
source 完成: free=8.19/23.64 GiB; allocated=14.97 GiB; peak=14.98 GiB


## 6. 逐个传播 target，并追踪真实 forward 参数
1 只接收 4，2 只接收 5，3 接收 4 和 5。追踪器打印实际传给模型的 dependency cache 长度、`position_ids` 和 `cache_position`；传播后只保存 target 新增部分。source 不应被原地追加 target token。


In [8]:
@torch.inference_mode()
def snapshot_sources():
    return {n:([x.clone() for x in gkv.nodes[n].key_cache],
               [x.clone() for x in gkv.nodes[n].value_cache]) for n in SOURCE_IDS}
before_sources = snapshot_sources()
trace = []
def trace_forward(module, args, kwargs):
    past = kwargs.get("past_key_values")
    record = {"past_len":0 if past is None else seq_len(past),
              "position_ids":kwargs["position_ids"].detach().cpu().tolist(),
              "cache_position":kwargs["cache_position"].detach().cpu().tolist()}
    trace.append(record)
hook = model.register_forward_pre_hook(trace_forward, with_kwargs=True)
try:
    for n in TARGET_IDS:
        deps = gkv.source_nodes_for(n)
        expected_past = sum(lengths[s] for s in deps)
        out = gkv.propagate_target(model, n, node_inputs[n].to(device))
        r = trace[-1]
        assert r["past_len"] == expected_past
        assert r["position_ids"][0] == list(range(P+L,P+L+lengths[n]))
        assert r["cache_position"] == list(range(expected_past,expected_past+lengths[n]))
        assert seq_len(out.past_key_values) == expected_past+lengths[n]
        assert seq_len(gkv.nodes[n]) == lengths[n]
        print(f"target {n} <- {deps}: dependency={expected_past}, "
              f"returned={seq_len(out.past_key_values)}, saved={seq_len(gkv.nodes[n])}")
        print("  logical:", (r["position_ids"][0][0],r["position_ids"][0][-1]+1),
              "physical:", (r["cache_position"][0],r["cache_position"][-1]+1))
        del out
finally:
    hook.remove()
for n,(keys,values) in before_sources.items():
    for old,new in zip(keys+values, gkv.nodes[n].key_cache+gkv.nodes[n].value_cache):
        assert torch.equal(old,new), f"source {n} 被修改"
del before_sources
gkv.validate()
assert gkv.graph == {"1":["4"], "3":["4","5"], "2":["5"]}
print("PASS: 三个 target 的依赖、位置、切片与 source 不变性")
display(gkv.summary())
memory("target 完成")


target 1 <- ['4']: dependency=20, returned=35, saved=15
  logical: (38, 53) physical: (20, 35)
target 2 <- ['5']: dependency=21, returned=37, saved=16
  logical: (38, 54) physical: (21, 37)
target 3 <- ['4', '5']: dependency=41, returned=59, saved=18
  logical: (38, 56) physical: (41, 59)
PASS: 三个 target 的依赖、位置、切片与 source 不变性


{'chunk_span': 21,
 'source_position_start': 17,
 'prefix_token_count': 17,
 'source_span': 21,
 'max_round': 1,
 'query_position_start': 59,
 'nodes': {'4': {'kind': 'source',
   'token_count': 20,
   'logical_range': (17, 37),
   'round': 0,
   'physical_seq_len': 20},
  '5': {'kind': 'source',
   'token_count': 21,
   'logical_range': (17, 38),
   'round': 0,
   'physical_seq_len': 21},
  '1': {'kind': 'target',
   'token_count': 15,
   'logical_range': (38, 53),
   'round': 1,
   'physical_seq_len': 15},
  '2': {'kind': 'target',
   'token_count': 16,
   'logical_range': (38, 54),
   'round': 1,
   'physical_seq_len': 16},
  '3': {'kind': 'target',
   'token_count': 18,
   'logical_range': (38, 56),
   'round': 1,
   'physical_seq_len': 18}},
 'edges': {'1': ['4'], '3': ['4', '5'], '2': ['5']}}

target 完成: free=8.18/23.64 GiB; allocated=14.98 GiB; peak=15.00 GiB


## 7. 独立数值参考：单次完整前向 + 显式图 mask
这一步使用 CPU 上的小型随机 Llama（无下载），不是 8B 问答结果。通过手写 dense 图 mask 与 v5 分块缓存比对，检查所有层的 K/V 与 query logits。它提供独立于缓存拼接的参考；随机模型不能评估回答质量。


In [9]:
from transformers import LlamaConfig, LlamaForCausalLM
@torch.inference_mode()
def check_dense_oracle():
    torch.manual_seed(101)
    cfg = LlamaConfig(vocab_size=97, hidden_size=32, intermediate_size=64,
        num_hidden_layers=3, num_attention_heads=4, num_key_value_heads=2,
        max_position_embeddings=256, attention_dropout=0.0)
    cfg._attn_implementation = "eager"
    tiny = LlamaForCausalLM(cfg).cpu().eval()
    make = lambda xs: torch.tensor([xs], dtype=torch.long)
    p = make([1,2]); q = make([30,31])
    sources = {"4":make([3,4,5]), "5":make([6,7])}
    targets = {"1":make([8,9]), "2":make([10,11,12]), "3":make([13,14,15,16])}
    pp = p.shape[1]; span = max(x.shape[1] for x in [*sources.values(),*targets.values()])
    chunks = [("prefix",p,0)] + [(n,x,pp) for n,x in sources.items()] + [(n,x,pp+span) for n,x in targets.items()] + [("query",q,pp+2*span)]
    ranges={}; positions=[]; offset=0
    for n,x,start in chunks:
        ranges[n]=(offset,offset+x.shape[1]); offset+=x.shape[1]
        positions.extend(range(start,start+x.shape[1]))
    allow=torch.zeros((offset,offset),dtype=torch.bool)
    for n,x,_ in chunks:
        a,b=ranges[n]
        allow[a:b,a:b]=torch.ones((b-a,b-a),dtype=torch.bool).tril()
        if n=="query": allow[a:b,:a]=True
        elif n in targets:
            for s,t in EDGES:
                if t==n:
                    c,d=ranges[s]; allow[a:b,c:d]=True
    mask=torch.zeros((1,1,offset,offset)).masked_fill(~allow,torch.finfo(torch.float32).min)
    reference=tiny(torch.cat([x for _,x,_ in chunks],-1),
        position_ids=torch.tensor([positions]),attention_mask=mask,use_cache=True,return_dict=True)
    actual=build_graphkv_from_ids(model=tiny,source_inputs=sources,target_inputs=targets,
        edges=EDGES,prefix_input_ids=p)
    for n in [*sources,*targets]:
        a,b=ranges[n]
        for attr in ["key_cache","value_cache"]:
            for x,y in zip(getattr(actual.nodes[n],attr),getattr(reference.past_key_values,attr)):
                torch.testing.assert_close(x,y[:,:,a:b,:],atol=2e-6,rtol=2e-5)
    logits=graphkv_next_logits(model=tiny,graph_cache=actual,query_input_ids=q)
    torch.testing.assert_close(logits,reference.logits[:,-1],atol=2e-6,rtol=2e-5)
    print("PASS: 五节点全部层 K/V 与 dense 图 mask 一致；query logits 最大差:",
          (logits-reference.logits[:,-1]).abs().max().item())
    return True
dense_oracle_passed = check_dense_oracle()


PASS: 五节点全部层 K/V 与 dense 图 mask 一致；query logits 最大差: 7.450580596923828e-08


## 8. 准备 query：拼接所有节点一次
物理顺序为 `[prefix,4,5,1,2,3]`，每个节点只出现一次。target 3 所依赖的 source 不应再作为 target 缓存的一部分重复加入。query 逻辑起点 P+2L，物理起点 P+所有节点 token 数之和。


In [10]:
query_cache, query_pos, query_slots = gkv.prepare_query(
    query_ids.to(device), source_ids=SOURCE_IDS, target_ids=TARGET_IDS)
physical_start = P+sum(lengths.values())
assert seq_len(query_cache) == physical_start
assert query_pos[0,0].item() == P+2*L
assert query_slots[0].item() == physical_start
print("query cache layer0 K shape:", tuple(query_cache.key_cache[0].shape))
print("物理 prefix+图长度:", physical_start, "逻辑 query 起点:", P+2*L)
print("query position_ids:", query_pos.cpu().tolist())
print("query cache_position:", query_slots.cpu().tolist())
print("已保存 prefix+node KV MiB:",
      (kv_bytes(gkv.prefix_cache)+sum(kv_bytes(c) for c in gkv.nodes.values()))/2**20)
del query_cache, query_pos, query_slots


query cache layer0 K shape: (1, 8, 107, 128)
物理 prefix+图长度: 107 逻辑 query 起点: 59
query position_ids: [[59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]]
query cache_position: [107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137]
已保存 prefix+node KV MiB: 13.375


## 9. 首 token 诊断与完整 greedy 生成
调用 v5 自带生成函数，逐步显式递增逻辑位置与物理位置。避免直接换成未验证的 `model.generate` 路径。EOS 可能造成合法空文本，因此同时保留原始 token 和停止原因。

预期语义答案：ORION=copper，LYRA=silver，NOVA=copper+silver。这只是人工预期，不保证模型严格按格式作答，也不能用语义回答证明图 mask 正确。query 可读取全部节点，所以它可能直接从 source 回答。


In [11]:
logits = graphkv_next_logits(model=model, graph_cache=gkv, query_input_ids=query_ids)
probs = logits.softmax(-1)
values, indices = probs[0].topk(5)
for token, prob in zip(indices.tolist(),values.tolist()):
    print(token, repr(tokenizer.decode([token],skip_special_tokens=False)), f"p={prob:.5f}")
del logits, probs, values, indices
before_summary = gkv.summary()
torch.cuda.synchronize(); started = time.perf_counter()
result = greedy_generate_graphkv(model=model, tokenizer=tokenizer, graph_cache=gkv,
    query_input_ids=query_ids, source_ids=SOURCE_IDS, target_ids=TARGET_IDS,
    max_new_tokens=MAX_NEW_TOKENS, return_details=True)
torch.cuda.synchronize(); elapsed = time.perf_counter()-started
assert gkv.summary() == before_summary
print("回答:", result.text)
print("raw token IDs:", result.token_ids)
print("含特殊 token 的文本:", repr(tokenizer.decode(result.token_ids,skip_special_tokens=False)))
print("stopped_on_eos:", result.stopped_on_eos, "seconds:", round(elapsed,3))
print("材料词检查（仅提示）:", {w:w in result.text.lower() for w in ["copper","silver"]})
gkv.validate()
memory("生成完成")


11439 'According' p=0.96519
29815 'Based' p=0.02003
878 'OR' p=0.01377
3915 'From' p=0.00047
10771 ' According' p=0.00017
回答: According to the provided documents:

* ORION uses the material supplied by BETA, which is silver.
* LYRA uses the material supplied by BETA, which is silver.
* NOVA combines the materials supplied by ALPHA (copper) and BETA (silver).
raw token IDs: [11439, 311, 279, 3984, 9477, 1473, 9, 2794, 1294, 5829, 279, 3769, 17665, 555, 426, 21352, 11, 902, 374, 15310, 627, 9, 87105, 5726, 5829, 279, 3769, 17665, 555, 426, 21352, 11, 902, 374, 15310, 627, 9, 5782, 13114, 33511, 279, 7384, 17665, 555, 83897, 320, 66, 18994, 8, 323, 426, 21352, 320, 83014, 570, 128001]
含特殊 token 的文本: 'According to the provided documents:\n\n* ORION uses the material supplied by BETA, which is silver.\n* LYRA uses the material supplied by BETA, which is silver.\n* NOVA combines the materials supplied by ALPHA (copper) and BETA (silver).<|end_of_text|>'
stopped_on_eos: True seconds: 1.444
材料

## 10. 保存本次真实诊断结果
重复运行生成会重新组装 query cache，可复用节点。JSON 不保存巨大的 K/V 张量，保留文本、图、位置、生成 token、版本和显存信息。


In [12]:
report = {
    "model_path":str(MODEL_PATH), "repo_dir":str(REPO_DIR),
    "reference_commit":"1af09232d361d8f529d951f35348664aad37d415",
    "torch":torch.__version__, "transformers":transformers.__version__,
    "physical_gpu":GPU_ID, "gpu_name":torch.cuda.get_device_name(0), "backend":BACKEND,
    "source_texts":source_texts, "target_texts":target_texts, "edges":EDGES,
    "prefix":PREFIX, "query":QUERY, "lengths":lengths, "P":P,"L":L,"Q":Q,
    "summary":gkv.summary(), "propagation_trace":trace,
    "dense_oracle_passed":dense_oracle_passed,
    "answer":result.text, "token_ids":result.token_ids,
    "stopped_on_eos":result.stopped_on_eos, "generation_seconds":elapsed,
    "peak_allocated_GiB":torch.cuda.max_memory_allocated()/2**30,
}
report_path = Path.cwd()/"graphkv_5nodes_single_gpu_results.json"
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print("已保存:", report_path)


已保存: /home/lyh/hcy_work/GraphKV_refactor_v5/graphkv_5nodes_single_gpu_results.json


## 常见问题
- 找不到模块：`REPO_DIR` 需要指向包含 `graph_kv_cache.py`、`graph_kv_adapter.py` 的内层目录；修改后重启内核，避免旧模块缓存。
- v5 对 Transformers 精确校验为 4.50.0。此 Notebook 不自动升级或覆盖已有环境。
- 空答案：查看 raw token、首 token 排名和 EOS 原因；不要直接据此认定缓存实现错误。
- 单卡 OOM：查看加载前空闲显存，释放旧模型或其他进程，再重启内核。此文件没有 CPU offload 或量化，确保所有模型参数确实在同一卡上。
- 逻辑跨度减少并不等于物理 KV token 数减少；query 仍持有全部选择的节点。

本文件交付时仅完成结构与代码静态校验；当前制作环境没有用户 Linux GPU、模型权重及 torch/transformers，因此未执行真实模型推理。第 7 步的数值校验也需在用户内核执行；只有实际打印 PASS 后才能记录为通过。
